In [1]:
from rdflib import Graph, Namespace, RDF, OWL, XSD, Literal
from rdflib.namespace import DCTERMS, SKOS
from pathlib import Path
import json

# Namespaces
SCHEMA = Namespace("http://schema.org/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
DANCE = Namespace("http://example.org/dance/")

# Graph setup
g = Graph()
g.bind("schema", SCHEMA)
g.bind("wd", WD)
g.bind("wdt", WDT)
g.bind("dance", DANCE)
g.bind("dcterms", DCTERMS)
g.bind("skos", SKOS)
g.bind("owl", OWL)
g.bind("xsd", XSD)


In [2]:
# Paths
project_root = Path("../..").resolve()
ontology_path = project_root / "kg_code/graphs/dance_ontology.ttl"
data_path = project_root / "data/ai_data" / "dance_dataset.jsonl"
output_path = project_root / "kg_code/graphs" / "KG_base_dance.ttl"

# Load ontology
if not ontology_path.exists():
    raise FileNotFoundError(f"Ontology not found: {ontology_path}")

g.parse(str(ontology_path), format="turtle")
print(f"Loaded ontology triples: {len(g)}")


Loaded ontology triples: 189


In [3]:


def ensure_named_instance(class_uri, value, prefix=None):

    key = str(value).strip().replace(" ", "_")
    node = DANCE[key]
    g.add((node, RDF.type, class_uri))
    g.add((node, SCHEMA.name, Literal(str(value), lang="en")))
    return node


In [4]:
# Ingest JSONL dance records
if not data_path.exists():
    raise FileNotFoundError(f"Data file not found: {data_path}")

record_count = 0
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)

        # Core record node
        rec_id = f"record_{row.get('dance_type', '')}_{row.get('dance_style', '')}".replace(" ", "_")
        record = DANCE[rec_id]
        g.add((record, RDF.type, DANCE.DanceRecord))
        g.add((record, SCHEMA.name, Literal(f"{row.get('dance_type', '')} - {row.get('dance_style', '')}", lang="en")))

        # Class-backed fields
        dance_type_node = ensure_named_instance(DANCE.DanceType, row.get("dance_type", "Unknown"))
        dance_style_node = ensure_named_instance(DANCE.DanceStyle, row.get("dance_style", "Unknown"))
        origin_node = ensure_named_instance(DANCE.Origin, row.get("origin", "Unknown"))
        period_node = ensure_named_instance(DANCE.TimePeriod, row.get("time_period", "Unknown"))

        g.add((record, DANCE.hasDanceType, dance_type_node))
        g.add((record, DANCE.hasDanceStyle, dance_style_node))
        g.add((record, DANCE.hasOrigin, origin_node))
        g.add((record, DANCE.hasTimePeriod, period_node))

        # Instruments (array)
        for inst in row.get("instruments", []):
            inst_node = ensure_named_instance(SCHEMA.MusicInstrument, inst)
            g.add((record, DANCE.hasInstrument, inst_node))

        # Formation / difficulty / age-group
        formation_node = ensure_named_instance(DANCE.DanceFormation, row.get("dance_formation", "Unknown"))
        difficulty_node = ensure_named_instance(DANCE.LearningDifficulty, row.get("learning_difficulty", "Unknown"))
        age_node = ensure_named_instance(DANCE.AgeGroup, row.get("age_group", "Unknown"))

        g.add((record, DANCE.hasDanceFormation, formation_node))
        g.add((record, DANCE.hasLearningDifficulty, difficulty_node))
        g.add((record, DANCE.hasAgeGroup, age_node))

        # Tempo
        tempo_value = row.get("tempo_bpm")
        if isinstance(tempo_value, dict):
            tmin = tempo_value.get("min")
            tmax = tempo_value.get("max")
        else:
            tmin = tempo_value
            tmax = tempo_value

        if tmin is not None and tmax is not None:
            tempo_node = DANCE[f"tempo_{rec_id}"]
            g.add((tempo_node, RDF.type, DANCE.TempoRange))
            g.add((tempo_node, DANCE.minTempoBPM, Literal(int(tmin), datatype=XSD.integer)))
            g.add((tempo_node, DANCE.maxTempoBPM, Literal(int(tmax), datatype=XSD.integer)))
            g.add((record, DANCE.hasTempoRange, tempo_node))

        # Scalar literal fields
        if row.get("cultural_significance"):
            g.add((record, DANCE.culturalSignificance, Literal(row["cultural_significance"], lang="en")))
        if row.get("notable_characteristics"):
            g.add((record, DANCE.notableCharacteristics, Literal(row["notable_characteristics"], lang="en")))
        if row.get("hardness_ratio") is not None:
            g.add((record, DANCE.hardnessRatio, Literal(str(row["hardness_ratio"]), datatype=XSD.decimal)))
        if row.get("costume"):
            g.add((record, DANCE.costume, Literal(row["costume"], lang="en")))
        if row.get("modern_adaptations"):
            g.add((record, DANCE.modernAdaptations, Literal(row["modern_adaptations"], lang="en")))

        # Arrays of class-backed entities
        for p in row.get("famous_practitioners", []):
            p_node = ensure_named_instance(DANCE.Practitioner, p)
            g.add((record, DANCE.hasFamousPractitioner, p_node))

        for e in row.get("events_and_festivals", []):
            e_node = ensure_named_instance(SCHEMA.Event, e)
            g.add((record, DANCE.hasEventFestival, e_node))

        genres = row.get("associated_music_genre", [])
        if isinstance(genres, str):
            genres = [genres]
        for genre in genres:
            genre_node = ensure_named_instance(SCHEMA.MusicGenre, genre)
            g.add((record, DANCE.hasAssociatedMusicGenre, genre_node))

        for hb in row.get("health_benefits", []):
            hb_node = ensure_named_instance(DANCE.HealthBenefit, hb)
            g.add((record, DANCE.hasHealthBenefit, hb_node))

        record_count += 1

print(f"Loaded records: {record_count}")
print(f"Total triples after data load: {len(g)}")


Loaded records: 130
Total triples after data load: 5908


In [5]:
#create yt ontology

In [6]:
# Save merged KG (ontology + instances)
g.serialize(destination=str(output_path), format="turtle")
print(f"Saved merged KG to: {output_path}")

# Quick sample
for i, triple in enumerate(g):
    if i >= 10:
        break
    print(triple)


Saved merged KG to: /Users/hannes/Documents/GitHub/KG_Personal_Project/kg_code/graphs/KG_base_dance.ttl
(rdflib.term.URIRef('http://example.org/dance/record_Swing_dance_Blues_dance'), rdflib.term.URIRef('http://example.org/dance/hasAssociatedMusicGenre'), rdflib.term.URIRef('http://example.org/dance/Blues'))
(rdflib.term.URIRef('http://example.org/dance/record_Street_dance_Vogue'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://example.org/dance/DanceRecord'))
(rdflib.term.URIRef('http://example.org/dance/record_Swing_dance_Bugg'), rdflib.term.URIRef('http://example.org/dance/costume'), rdflib.term.Literal('Casual or semi-formal attire suitable for social dancing, often comfortable shoes for quick footwork.', lang='en'))
(rdflib.term.URIRef('http://example.org/dance/record_Swing_dance_Black_Bottom'), rdflib.term.URIRef('http://example.org/dance/hasInstrument'), rdflib.term.URIRef('http://example.org/dance/Piano'))
(rdflib.term.URIRef('